In [1]:
import requests
import os
import zipfile
import plotly.express as px
import pandas as pd
from pyspark.sql.functions import (
    col, to_date, initcap, when,
    countDistinct, sum as _sum, mean as _mean,
    concat_ws, lit
)
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

# https://portaldatransparencia.gov.br/pagina-interna/605543-dicionario-de-dados-novo-bolsa-familia
# https://portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia


ANO = "2026"
MES = "01"
ANO_MES = f"{ANO}{MES}"

URL_DOWNLOAD = f"https://portaldatransparencia.gov.br/download-de-dados/novo-bolsa-familia/{ANO_MES}"

PATH_DESTINO_ZIP = f"/lakehouse/default/Files/raw_bolsa_familia/{ANO_MES}.zip"
PATH_EXTRAIDO = f"/lakehouse/default/Files/raw_bolsa_familia/{ANO_MES}/"

# EXTRAÇÃO

def baixar_arquivo(url, caminho_salvar):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    # stream=True é crucial para arquivos grandes (não carrega tudo na RAM de uma vez)
    with requests.get(url, headers=headers, stream=True) as r:
        r.raise_for_status() # Para se der erro 404/500
        with open(caminho_salvar, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192): 
                f.write(chunk)
    
    print(f"Download concluído: {caminho_salvar}")


def descompactar(caminho_zip, caminho_destino):
    with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
        zip_ref.extractall(caminho_destino)
    print(f"Arquivos extraídos em: {caminho_destino}")


try:
    print(f"Iniciando download de {ANO_MES}...")
    baixar_arquivo(URL_DOWNLOAD, PATH_DESTINO_ZIP)
    
    print("Extraindo arquivos...")
    descompactar(PATH_DESTINO_ZIP, PATH_EXTRAIDO)
    
    # Remover o ZIP da pasta Files
    os.remove(PATH_DESTINO_ZIP)
    print("Processo finalizado com sucesso.")
    
except Exception as e:
    print(f"Erro durante o processo: {e}")


# TRANSFORMAÇÃO E LOAD
def ajustar_nome_coluna(nome_coluna):
    nome_coluna = nome_coluna.lower().replace(" ", "_")
    nome_coluna = (
        nome_coluna.replace("ê", "e")
        .replace("ç", "c")
        .replace("ã", "a")
        .replace("í", "i")
    )
    return nome_coluna


schema = StructType([
    StructField("MÊS COMPETÊNCIA", StringType(), True),
    StructField("MÊS REFERÊNCIA", StringType(), True),
    StructField("UF", StringType(), True),
    StructField("CÓDIGO MUNICÍPIO SIAFI", StringType(), True),
    StructField("NOME MUNICÍPIO", StringType(), True),
    StructField("CPF FAVORECIDO", StringType(), True),
    StructField("NIS FAVORECIDO", StringType(), True),
    StructField("NOME FAVORECIDO", StringType(), True),
    StructField("VALOR PARCELA", StringType(), True),
])

df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("encoding", "latin1")
    .option("sep", ";")
    .schema(schema)
    .load(f"Files/raw_bolsa_familia/{ANO_MES}/*.csv")
)

df_sp = df.filter(df["UF"] == "SP")
df_sp = df_sp.select([col(c).alias(ajustar_nome_coluna(c)) for c in df_sp.columns])
df_sp = df_sp.withColumn(
    "valor_parcela", F.regexp_replace(F.col("valor_parcela"), ",", ".").cast("double")
)
df_sp = (
    df_sp
    .withColumn(
        "mes_referencia",
        to_date(concat_ws("", col("mes_referencia").cast("string"), lit("01")), "yyyyMMdd")
    )
    .withColumn(
        "mes_competencia",
        to_date(concat_ws("", col("mes_competencia").cast("string"), lit("01")), "yyyyMMdd")
    )
)

(
    df_sp
    .write.format("delta")
    .mode("append")
    .saveAsTable("lh_cidade_inteligente_osasco.silver_pbf_sp")
)
print("Tabela escrita com sucesso.")

StatementMeta(, fdd4780f-8ae8-404d-8e42-a9dda1ff8b43, 3, Finished, Available, Finished, False)

Iniciando download de 202601...
Download concluído: /lakehouse/default/Files/raw_bolsa_familia/202601.zip
Extraindo arquivos...
Arquivos extraídos em: /lakehouse/default/Files/raw_bolsa_familia/202601/
Processo finalizado com sucesso.
Tabela escrita com sucesso.
